Copyright (c) European Space Agency, 2026.  
This file is subject to the terms and conditions defined in file 'LICENCE.txt',   
which is part of this source code package. No part of the package, including  
this file, may be copied, modified, propagated, or distributed except according to   
the terms contained in the file ‘LICENCE.txt’.

# Download the pre-release epoch_astrometry dataset from the public FTP server and analyse it using `gaiasupdate`

The `gaiasupdate` package can be found at https://github.com/esa/gaia-supdate.  
It can be installed with `pip install gaiasupdate`

In [15]:
import logging
import urllib.request
import zipfile
import os
import glob

import numpy as np
import pandas as pd

from gaiasupdate.epoch_astrometry import GaiaEpochAstrometryArchive
from astropy.table import Table

logger = logging.getLogger()
logger.setLevel(logging.INFO)

## Download astrometric-timeseries data

In [16]:
# URL of the file to download
url = "https://anonftp.cosmos.esa.int/pub/GAIA_PUBLIC_DATA/Gaia_DR4/dr4-prerelease/gaia-dr4-prerelease-epoch-astrometry_2026-06-26.zip"

local_dir = os.getcwd()

# Define download path
download_path = os.path.join(local_dir, 'downloaded_file.zip')
extract_path = os.path.join(local_dir, 'extracted_files')

if not os.path.isfile(download_path):

    # Download the file
    urllib.request.urlretrieve(url, os.path.basename(download_path))
    logging.info(f"Downloaded {url} to {os.path.basename(download_path)}")

with zipfile.ZipFile(download_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
logging.info(f"Extracted {os.path.basename(download_path)} to {os.path.basename(extract_path)}")

INFO:root:Extracted downloaded_file.zip to extracted_files


In [17]:
epoch_astrometry_file = glob.glob(os.path.join(extract_path, '*.xml'))[0]
logging.info(f"Found epoch astrometry file: {os.path.basename(epoch_astrometry_file)}")

INFO:root:Found epoch astrometry file: GAIA_DR4_PRERELEASE_EPOCH_ASTROMETRY_RAW.xml


In [18]:
# Read VOTable with astropy.table and convert to pandas
table = Table.read(epoch_astrometry_file, format="votable")
df = table.to_pandas()
logging.info(f"File contains data for {df['source_id'].nunique()} unique sources.\n")
display(df.info())

INFO:root:File contains data for 12 unique sources.



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   solution_id                1008 non-null   int64  
 1   source_id                  1008 non-null   int64  
 2   transit_id                 1008 non-null   int64  
 3   ra0                        1008 non-null   float64
 4   dec0                       1008 non-null   float64
 5   agis_source_excess_noise   1008 non-null   float32
 6   obs_time_tcb               1008 non-null   object 
 7   obs_time_bary_corr         911 non-null    float32
 8   scan_pos_angle             1008 non-null   object 
 9   zeta                       911 non-null    float32
 10  parallax_factor_al         911 non-null    float32
 11  parallax_factor_ac         911 non-null    float32
 12  colour_factor_al           1008 non-null   object 
 13  colour_factor_ac           1008 non-null   objec

None

In [19]:
# helper function 
def map_results(sourceid, supdate):

    supdate_results = {}
    params = ['deltaAlphaStar_mas', 'deltaDelta_mas', 'varpi_mas', 'muAlphaStar_maspyr', 'muDelta_maspyr', 'pseudoColor_offset']
    
    supdate_results['sourceId'] = sourceid
    supdate_results['success'] = np.where(np.isnan(supdate['parameters'][2]), 0, 1).item()
    supdate_results['paramsSolved'] = np.where(supdate['model'] == '6p_constrained_colour', 95, 31).item()
    
    agis_ppm_keys = [ f'{s}' for s in params ]
    agis_ppm_error_keys = [ f'{p}Error' for p in params ]
                
    map_source_to_results = {'excessNoise': 'excess_noise', 'excessNoiseSig': 'significance' }
    map_source_to_statistic = { 'f2': 'f2', 'chi2Al': 'chi2', 'nObsAl': 'n_measurements', 'nOutliersAl': 'n_outliers' }

    for field, mapped_field in map_source_to_results.items(): supdate_results[field] = supdate[mapped_field]
        
    for field, mapped_field in map_source_to_statistic.items(): supdate_results[field] = getattr(supdate['solution_statistic'], mapped_field)
        
    for i, key in enumerate(agis_ppm_error_keys): supdate_results[key] = supdate['parameters_formal_uncertainty'][i]

    for i, key in enumerate(agis_ppm_keys): supdate_results[key] = supdate['parameters'][i]

    return supdate_results

## Perform the single-star model fit in the same way as the core astrometric solution of Gaia DR4, but on epoch_astrometry data

The output parameters of the `GaiaEpochAstrometryArchive.supdate` method are described at https://esa.github.io/gaia-supdate/supdate_output_params.html 

In [20]:
epoch_astro = df.copy()

for source_id in df['source_id'].unique():
    logging.info(f"Processing source_id: {source_id}")
    res_6p = GaiaEpochAstrometryArchive.supdate(epoch_astro, source_id)

    res_6p_df = pd.DataFrame( [map_results(source_id, res_6p)] )
    display(res_6p_df[['deltaAlphaStar_mas', 'deltaDelta_mas', 'varpi_mas', 'muAlphaStar_maspyr', 'muDelta_maspyr', 'pseudoColor_offset', 'paramsSolved']])
    


INFO:root:Processing source_id: 4181040337841125632
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on usedByAgisAl==True removed 333 entries.
INFO:root:Filter on column obsTimeBaryCorr!=null removed 0 entries.
INFO:root:Filter on column scanPosAngle!=null removed 0 entries.
INFO:root:GaiaSourceEpochAstrometryCu9 for source_id 4181040337841125632 with 637 CCD         transits


,deltaAlphaStar_mas,deltaDelta_mas,varpi_mas,muAlphaStar_maspyr,muDelta_maspyr,pseudoColor_offset,paramsSolved
0,-0.000002,4.894602e-08,1.001305,2.903832,-0.294277,0.000002,95


INFO:root:Processing source_id: 435469040545191680
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on usedByAgisAl==True removed 161 entries.
INFO:root:Filter on column obsTimeBaryCorr!=null removed 0 entries.
INFO:root:Filter on column scanPosAngle!=null removed 0 entries.
INFO:root:GaiaSourceEpochAstrometryCu9 for source_id 435469040545191680 with 649 CCD         transits


,deltaAlphaStar_mas,deltaDelta_mas,varpi_mas,muAlphaStar_maspyr,muDelta_maspyr,pseudoColor_offset,paramsSolved
0,-0.000013,-0.000008,0.997734,3.421709,-5.812479,-3.402614e-07,95


INFO:root:Processing source_id: 3926186255616949504
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on usedByAgisAl==True removed 348 entries.
INFO:root:Filter on column obsTimeBaryCorr!=null removed 0 entries.
INFO:root:Filter on column scanPosAngle!=null removed 0 entries.
INFO:root:GaiaSourceEpochAstrometryCu9 for source_id 3926186255616949504 with 462 CCD         transits


...

,deltaAlphaStar_mas,deltaDelta_mas,varpi_mas,muAlphaStar_maspyr,muDelta_maspyr,pseudoColor_offset,paramsSolved
0,-0.000022,-0.000339,1.960751,-30.739616,-148.926288,0.00003,95


INFO:root:Processing source_id: 3937211745905473024
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on usedByAgisAl==True removed 342 entries.
INFO:root:Filter on column obsTimeBaryCorr!=null removed 0 entries.
INFO:root:Filter on column scanPosAngle!=null removed 0 entries.
INFO:root:GaiaSourceEpochAstrometryCu9 for source_id 3937211745905473024 with 558 CCD         transits


,deltaAlphaStar_mas,deltaDelta_mas,varpi_mas,muAlphaStar_maspyr,muDelta_maspyr,pseudoColor_offset,paramsSolved
0,0.001366,-0.000123,25.631659,-582.029241,-0.708667,0.000014,95


INFO:root:Processing source_id: 1457486023639239296
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on column obsTimeTcb!=null removed 0 entries.
INFO:root:Filter on usedByAgisAl==True removed 326 entries.
INFO:root:Filter on column obsTimeBaryCorr!=null removed 0 entries.
INFO:root:Filter on column scanPosAngle!=null removed 0 entries.
INFO:root:GaiaSourceEpochAstrometryCu9 for source_id 1457486023639239296 with 824 CCD         transits


,deltaAlphaStar_mas,deltaDelta_mas,varpi_mas,muAlphaStar_maspyr,muDelta_maspyr,pseudoColor_offset,paramsSolved
0,-0.000037,0.000009,13.624072,-75.552329,17.943998,-5.029289e-07,95
